# LeetCode 2316: Count Unreachable Pairs of Nodes in an Undirected Graph

## Approach: DSU + Component Size + Greedy Counting

### 1. Algorithm

1. Initialize `parent[i] = i` and `rank[i] = 0`.
2. Traverse all edges and perform **Union** to connect nodes belonging to the same component.
3. Use `find()` to identify the representative of each node.
4. Store component sizes in an `unordered_map`.
   - `{component_parent, component_size}`
5. Count unreachable pairs using:

   `pairs = size * (remaining - size)`

6. Initially, `remaining = n`.
7. After processing each component, update:

   `remaining -= size`

8. Return `ans`.

### 2. Core Logic

```cpp
for(auto &s : mp) {
    long long size = s.second;

    ans += size * (rem - size);
    rem -= size;
}
```

- `size` → Current component size.
- `rem` → Number of nodes not yet processed.
- `rem - size` → Nodes belonging to other remaining components.
- Multiplication counts pairs between the current component and other components.

**Why update `rem`?**

To avoid counting the same pair twice.

### 3. Dry Run

**Input:**

```cpp
n = 7;

edges = {
    {0,2}, {0,5}, {2,4},
    {1,6}, {5,4}
};
```

**Connected Components:**

```text
Component 1: {0, 2, 4, 5}  → Size = 4

Component 2: {1, 6}       → Size = 2

Component 3: {3}          → Size = 1
```

**Component Size Map:**

```text
{parent, size}

{0, 4}
{1, 2}
{3, 1}
```

| Component size | Nodes remaining | Pairs added | Nodes remaining after |
|:--|--:|--:|--:|
| 4 | 7 | 12 | 3 |
| 2 | 3 | 2 | 1 |
| 1 | 1 | 0 | 0 |

**Final Answer = 12 + 2 + 0 = 14**

### 4. Complexity Analysis

Let:
- `V` = Number of vertices
- `E` = Number of edges

| Operation | Complexity |
|---|---|
| DSU Initialization | O(V) |
| Union Operations | O(E × α(V)) |
| Component Size Mapping | O(V × α(V)) |
| Pair Counting | O(V) |

**Overall Time Complexity:**

`O(V + E × α(V))`

**Space Complexity:**

`O(V)`

`α(V)` = Inverse Ackermann Function (practically constant).

### 5. Remember

- Same representative → Same connected component.
- Different representatives → Different components.
- Use `unordered_map` to calculate component sizes.
- `size × (remaining - size)` counts unreachable pairs.
- Update `remaining` after processing each component to avoid duplicate counting.

---

### 6. Complete Code

```cpp
class Solution {
public:
    int find(int u, vector<int>& parent) {
        if (parent[u] == u)
            return u;

        return parent[u] = find(parent[u], parent);
    }

    void Union(int u, int v, vector<int>& parent, vector<int>& rank) {
        int x = find(u, parent);
        int y = find(v, parent);

        if (x == y)
            return;

        if (rank[x] > rank[y])
            parent[y] = x;

        else if (rank[x] < rank[y])
            parent[x] = y;

        else {
            parent[y] = x;
            rank[x]++;
        }
    }

    long long countPairs(int n, vector<vector<int>>& edges) {

        int E = edges.size();

        vector<int> parent(n);
        vector<int> rank(n, 0);

        for (int i = 0; i < n; i++) {
            parent[i] = i;
        }

        unordered_map<int, int> mp;

        long long rem = n;
        long long ans = 0;

        // 1. Make components
        for (int i = 0; i < E; i++) {
            int u = edges[i][0];
            int v = edges[i][1];

            Union(u, v, parent, rank);
        }

        // 2. Calculate component sizes
        for (int i = 0; i < n; i++) {
            int i_parent = find(i, parent);
            mp[i_parent]++;
        }

        // 3. Count unreachable pairs
        for (auto& s : mp) {
            long long size = s.second;

            ans += size * (rem - size);
            rem -= size;
        }

        return ans;
    }
};
```